In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sklearn
import shap
import os
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).resolve()
GOLD_DIR = DATA_DIR / "gold"
# Analysis outputs
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHARTS_DIR = OUTPUT_DIR / "charts"
METRICS_DIR = OUTPUT_DIR / "metrics"

# Create output directories
for directory in [OUTPUT_DIR, CHARTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: D:\Projects\remarcable-assignment
Data directory: D:\Projects\remarcable-assignment\data


In [3]:
dim_customer = pd.read_csv(
    GOLD_DIR / "dim_customer.csv"
)

dim_item = pd.read_csv(
    GOLD_DIR / "dim_item.csv"
)

dim_vendor = pd.read_csv(
    GOLD_DIR / "dim_vendor.csv"
)

fct_purchase_order_line = pd.read_csv(
    GOLD_DIR / "fct_purchase_order_line.csv"
)

item_purchase_history = pd.read_csv(
    GOLD_DIR / "item_purchase_history.csv"
)

validated_purchase_order_lines = pd.read_csv(
    GOLD_DIR / "fct_purchase_order_line_validated.csv"
)

In [4]:
# ============================================
# CUSTOMER DATA SUMMARY
# ============================================

customer_summary = (
    validated_purchase_order_lines
    .groupby('customer_id')
    .agg(
        Purchase_Lines=('po_number', 'size'),
        Unique_Purchase_Orders=('po_number', 'nunique'),
        Unique_Items=('item_id', 'nunique'),
        Unique_Vendors=('vendor_id', 'nunique'),
        First_Order_Date=('order_date', 'min'),
        Last_Order_Date=('order_date', 'max')
    )
    .sort_values('Purchase_Lines', ascending=False)
)

display(customer_summary)

print("Total customers:",
      validated_purchase_order_lines['customer_id'].nunique())

print("Total purchase lines:",
      len(validated_purchase_order_lines))

,Purchase_Lines,Unique_Purchase_Orders,Unique_Items,Unique_Vendors,First_Order_Date,Last_Order_Date
customer_id,,,,,,
CUST-A,2315,820,122,12,2024-01-01,2026-09-01
CUST-B,1583,567,120,12,2024-01-02,2026-09-01
CUST-D,1409,494,120,12,2024-01-03,2026-08-28
CUST-F,819,281,113,12,2024-01-05,2026-08-31
CUST-C,772,276,115,12,2024-01-11,2026-08-27
CUST-H,698,256,109,12,2024-01-04,2026-09-01
CUST-E,466,161,101,12,2024-01-04,2026-09-01
CUST-G,400,144,94,12,2024-01-06,2026-08-30


Total customers: 8
Total purchase lines: 8462


In [5]:
# ============================================
# ITEM OVERLAP BETWEEN CUSTOMERS
# ============================================

customer_item_matrix = pd.crosstab(
    validated_purchase_order_lines['item_id'],
    validated_purchase_order_lines['customer_id']
)

customer_item_counts = (
    customer_item_matrix.gt(0).sum()
)

print("Items purchased by each customer:")
display(customer_item_counts.to_frame('Unique_Items'))

print("\nItems purchased by multiple customers:")

shared_items = customer_item_matrix[
    customer_item_matrix.gt(0).sum(axis=1) > 1
]

display(shared_items.head(30))

Items purchased by each customer:


,Unique_Items
customer_id,
CUST-A,122
CUST-B,120
CUST-C,115
CUST-D,120
CUST-E,101
CUST-F,113
CUST-G,94
CUST-H,109



Items purchased by multiple customers:


customer_id,CUST-A,CUST-B,CUST-C,CUST-D,CUST-E,CUST-F,CUST-G,CUST-H
item_id,,,,,,,,
ITM-0104,11,25,7,5,1,3,3,2
ITM-0201,90,18,24,13,11,10,6,5
ITM-0208,7,10,3,5,0,5,6,4
ITM-0211,3,8,5,6,2,2,0,1
ITM-0219,3,4,2,8,0,1,0,0
ITM-0220,43,78,34,28,8,18,13,13
ITM-0224,11,4,2,21,1,2,1,4
ITM-0225,4,10,10,3,0,5,1,4
ITM-0228,20,67,11,59,8,10,11,5


In [6]:
# ============================================
# VENDOR OVERLAP
# ============================================

customer_vendor_matrix = pd.crosstab(
    validated_purchase_order_lines['vendor_id'],
    validated_purchase_order_lines['customer_id']
)

shared_vendors = customer_vendor_matrix[
    customer_vendor_matrix.gt(0).sum(axis=1) > 1
]

print("Vendors shared between customers:")
display(shared_vendors)

Vendors shared between customers:


customer_id,CUST-A,CUST-B,CUST-C,CUST-D,CUST-E,CUST-F,CUST-G,CUST-H
vendor_id,,,,,,,,
VEND-01,184,142,53,134,52,96,43,48
VEND-02,199,163,81,143,36,49,30,52
VEND-03,202,116,50,92,32,64,8,58
VEND-04,162,130,50,128,49,45,42,35
VEND-05,167,138,82,109,34,78,20,53
VEND-06,172,138,54,118,32,94,18,60
VEND-07,216,146,52,139,39,77,31,93
VEND-08,194,115,64,87,50,66,59,52
VEND-09,241,133,72,77,22,73,20,52


In [7]:

# ============================================
# 1. Start with all active customer transactions
# ============================================

model_data = validated_purchase_order_lines[
    validated_purchase_order_lines["status"].isin(
        ["Received", "Closed", "Open"]
    )
].copy()

model_data = model_data.dropna(
    subset=['unit_price_paid']
).copy()

model_data['order_date'] = pd.to_datetime(
    model_data['order_date']
)

# Sort chronologically
model_data = model_data.sort_values(
    ['item_id', 'order_date', 'po_number']
).copy()

print("Total modelling records:", len(model_data))
print("Unique customers:", model_data['customer_id'].nunique())


# ============================================
# 2. Calendar features
# ============================================

model_data['year'] = model_data['order_date'].dt.year
model_data['month'] = model_data['order_date'].dt.month
model_data['quarter'] = model_data['order_date'].dt.quarter


# ============================================
# 3. Previous paid price
# ============================================

model_data['previous_paid_price'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .shift(1)
)


# ============================================
# 4. Previous 3-order average
# ============================================

model_data['previous_3_order_avg'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)


# ============================================
# 5. Previous 3-order median
# ============================================

model_data['previous_3_order_median'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=1
        ).median()
    )
)


# ============================================
# 6. Historical item average across vendors
# ============================================

model_data['historical_item_avg'] = (
    model_data.groupby('item_id')
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)


# ============================================
# 7. Historical item median across vendors
# ============================================

model_data['historical_item_median'] = (
    model_data.groupby('item_id')
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().median()
    )
)


# ============================================
# 8. Historical item-vendor average
# ============================================

model_data['historical_vendor_item_avg'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)


# ============================================
# 9. Historical item-vendor median
# ============================================

model_data['historical_vendor_item_median'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().median()
    )
)


# ============================================
# 10. Historical item-vendor history indicator
# ============================================

model_data['has_previous_vendor_item_history'] = (
    model_data['historical_vendor_item_avg']
    .notna()
    .astype(int)
)


# ============================================
# 11. Historical item transaction count
# ============================================

model_data['historical_item_count'] = (
    model_data.groupby('item_id')
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().count()
    )
)


# ============================================
# 12. Historical vendor-item transaction count
# ============================================

model_data['historical_vendor_item_count'] = (
    model_data.groupby(['item_id', 'vendor_id'])
    ['unit_price_paid']
    .transform(
        lambda x: x.shift(1).expanding().count()
    )
)


# ============================================
# 13. Inspect resulting dataset
# ============================================

display(model_data.head(15))


# ============================================
# 14. Check missing values
# ============================================

engineered_features = [
    'previous_paid_price',
    'previous_3_order_avg',
    'previous_3_order_median',
    'historical_item_avg',
    'historical_item_median',
    'historical_item_count',
    'historical_vendor_item_avg',
    'historical_vendor_item_median',
    'historical_vendor_item_count',
    'has_previous_vendor_item_history'
]

display(
    model_data[engineered_features].isna().sum()
)


# ============================================
# 15. Validate customer coverage
# ============================================

print("Total records:", len(model_data))
print("Unique customers:", model_data['customer_id'].nunique())
print("Unique items:", model_data['item_id'].nunique())
print("Unique vendors:", model_data['vendor_id'].nunique())

display(
    model_data.groupby('customer_id')
    .size()
    .to_frame('Purchase_Lines')
)


Total modelling records: 7905
Unique customers: 8


,po_number,customer_id,vendor_id,order_date,status,item_id,quantity_ordered,unit_price_paid,extended_amount,year,month,quarter,previous_paid_price,previous_3_order_avg,previous_3_order_median,historical_item_avg,historical_item_median,historical_vendor_item_avg,historical_vendor_item_median,has_previous_vendor_item_history,historical_item_count,historical_vendor_item_count
7166,PO-102710,CUST-B,VEND-03,2024-01-03,Closed,ITM-0104,6,64.99,389.94,2024,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.00,0.00
4966,PO-101883,CUST-C,VEND-11,2024-01-17,Received,ITM-0104,6,61.88,371.28,2024,1,1,NaN,NaN,NaN,64.99,64.99,NaN,NaN,0,1.00,0.00
5433,PO-102045,CUST-B,VEND-07,2024-01-27,Closed,ITM-0104,1,61.19,61.19,2024,1,1,NaN,NaN,NaN,63.44,63.44,NaN,NaN,0,2.00,0.00
2124,PO-100801,CUST-D,VEND-03,2024-02-26,Received,ITM-0104,1,65.81,65.81,2024,2,1,64.99,64.99,64.99,62.69,61.88,64.99,64.99,1,3.00,1.00
1273,PO-100462,CUST-C,VEND-03,2024-03-22,Closed,ITM-0104,4,71.07,284.28,2024,3,1,65.81,65.40,65.40,63.47,63.44,65.40,65.40,1,4.00,2.00
8429,PO-103171,CUST-B,VEND-07,2024-06-08,Received,ITM-0104,10,66.28,662.80,2024,6,2,61.19,61.19,61.19,64.99,64.99,61.19,61.19,1,5.00,1.00
1184,PO-100429,CUST-C,VEND-03,2024-06-26,Received,ITM-0104,10,72.44,724.40,2024,6,2,71.07,67.29,65.81,65.20,65.40,67.29,65.81,1,6.00,3.00
4719,PO-101785,CUST-B,VEND-03,2024-07-14,Received,ITM-0104,2,73.55,147.10,2024,7,3,72.44,69.77,71.07,66.24,65.81,68.58,68.44,1,7.00,4.00
2889,PO-101076,CUST-A,VEND-06,2024-07-21,Received,ITM-0104,6,66.72,400.32,2024,7,3,NaN,NaN,NaN,67.15,66.05,NaN,NaN,0,8.00,0.00
3089,PO-101148,CUST-B,VEND-06,2024-07-30,Received,ITM-0104,2,70.75,141.50,2024,7,3,66.72,66.72,66.72,67.10,66.28,66.72,66.72,1,9.00,1.00


previous_paid_price                 382
previous_3_order_avg                382
previous_3_order_median             382
historical_item_avg                 129
historical_item_median              129
historical_item_count                 0
historical_vendor_item_avg          382
historical_vendor_item_median       382
historical_vendor_item_count          0
has_previous_vendor_item_history      0
dtype: int64

Total records: 7905
Unique customers: 8
Unique items: 129
Unique vendors: 12


,Purchase_Lines
customer_id,
CUST-A,2178
CUST-B,1475
CUST-C,709
CUST-D,1308
CUST-E,443
CUST-F,762
CUST-G,387
CUST-H,643


In [8]:
# ============================================
# ALL CUSTOMERS: ACTIVE PURCHASE RECORDS
# ============================================

model_data["order_date"] = pd.to_datetime(
    model_data["order_date"]
)

model_data = model_data.sort_values(
    ['order_date', 'po_number', 'item_id']
).copy()


# ============================================
# VERIFY CUSTOMER DISTRIBUTION
# ============================================

print("Total modelling population:", len(model_data))

print(
    "Number of customers:",
    model_data["customer_id"].nunique()
)

display(
    model_data.groupby("customer_id").size()
    .sort_values(ascending=False)
    .to_frame("Purchase_Lines")
)

Total modelling population: 7905
Number of customers: 8


,Purchase_Lines
customer_id,
CUST-A,2178
CUST-B,1475
CUST-D,1308
CUST-F,762
CUST-C,709
CUST-H,643
CUST-E,443
CUST-G,387


In [9]:
# ============================================
# CHRONOLOGICAL TRAIN/TEST SPLIT
# ============================================

model_data = model_data.sort_values(
    ['order_date', 'po_number', 'item_id']
).reset_index(drop=True)

split_idx = int(len(model_data) * 0.80)

split_date = model_data.iloc[split_idx]["order_date"]

train_data = model_data[
    model_data["order_date"] < split_date
].copy()

test_data = model_data[
    model_data["order_date"] >= split_date
].copy()

print("Split date:", split_date)

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

print(
    "Training period:",
    train_data["order_date"].min(),
    "to",
    train_data["order_date"].max()
)

print(
    "Testing period:",
    test_data["order_date"].min(),
    "to",
    test_data["order_date"].max()
)

Split date: 2026-02-26 00:00:00
Training records: 6323
Testing records: 1582
Training period: 2024-01-01 00:00:00 to 2026-02-25 00:00:00
Testing period: 2026-02-26 00:00:00 to 2026-09-01 00:00:00


In [10]:
# ============================================
# CUSTOMER A TEST POPULATION
# ============================================

company_a_test = test_data[
    test_data["customer_id"] == "CUST-A"
].copy()

print(
    "CUST-A test records:",
    len(company_a_test)
)

print(
    "CUST-A test purchase orders:",
    company_a_test["po_number"].nunique()
)

display(company_a_test.head())

CUST-A test records: 416
CUST-A test purchase orders: 150


,po_number,customer_id,vendor_id,order_date,status,item_id,quantity_ordered,unit_price_paid,extended_amount,year,month,quarter,previous_paid_price,previous_3_order_avg,previous_3_order_median,historical_item_avg,historical_item_median,historical_vendor_item_avg,historical_vendor_item_median,has_previous_vendor_item_history,historical_item_count,historical_vendor_item_count
6329,PO-101654,CUST-A,VEND-07,2026-02-26,Closed,ITM-0306,4,0.99,3.96,2026,2,1,1.02,1.04,1.04,0.96,0.96,0.98,0.98,1,176.00,49.00
6330,PO-101654,CUST-A,VEND-07,2026-02-26,Closed,ITM-0612,12,4.05,48.60,2026,2,1,3.98,4.15,4.06,4.06,4.04,4.14,4.14,1,240.00,86.00
6338,PO-102980,CUST-A,VEND-05,2026-02-27,Received,ITM-0229,24,1.45,34.80,2026,2,1,1.70,1.56,1.49,1.55,1.56,1.54,1.48,1,34.00,4.00
6339,PO-102980,CUST-A,VEND-05,2026-02-27,Received,ITM-0426,1,24.19,24.19,2026,2,1,25.97,24.72,25.34,24.96,24.99,24.12,24.21,1,193.00,40.00
6340,PO-102980,CUST-A,VEND-05,2026-02-27,Received,ITM-0433,1,1.09,1.09,2026,2,1,1.09,1.07,1.07,1.19,1.19,1.09,1.09,1,241.00,45.00


In [11]:
assert train_data['order_date'].max() < test_data['order_date'].min()

print("Chronological split validated.")

Chronological split validated.


In [12]:
# ============================================
# CUSTOMER DISTRIBUTION
# ============================================

customer_split_summary = pd.DataFrame({
    "Training": train_data.groupby("customer_id").size(),
    "Testing": test_data.groupby("customer_id").size()
}).fillna(0).astype(int)

customer_split_summary["Total"] = (
    customer_split_summary["Training"] +
    customer_split_summary["Testing"]
)

display(customer_split_summary)

,Training,Testing,Total
customer_id,,,
CUST-A,1762,416,2178
CUST-B,1186,289,1475
CUST-C,568,141,709
CUST-D,1011,297,1308
CUST-E,363,80,443
CUST-F,610,152,762
CUST-G,304,83,387
CUST-H,519,124,643


In [13]:
# ============================================
# CUST-A TEST DATA
# ============================================

company_a_test = test_data[
    test_data["customer_id"] == "CUST-A"
].copy()

print("CUST-A test records:", len(company_a_test))
print(
    "CUST-A unique items:",
    company_a_test["item_id"].nunique()
)

print(
    "CUST-A unique vendors:",
    company_a_test["vendor_id"].nunique()
)

CUST-A test records: 416
CUST-A unique items: 100
CUST-A unique vendors: 12


In [14]:
FEATURES = [
    'item_id',
    'vendor_id',
    'quantity_ordered',
    'year',
    'month',
    'quarter',

    # Previous price features
    'previous_paid_price',
    'previous_3_order_avg',
    'previous_3_order_median',

    # Historical item pricing
    'historical_item_avg',
    'historical_item_median',
    'historical_item_count',

    # Historical vendor-item pricing
    'historical_vendor_item_avg',
    'historical_vendor_item_median',
    'historical_vendor_item_count',

    # History indicator
    'has_previous_vendor_item_history'
]

TARGET = 'unit_price_paid'

In [15]:
# Convert date
model_data['order_date'] = pd.to_datetime(
    model_data['order_date']
)

# Sort chronologically
model_data = model_data.sort_values(
    ['order_date', 'po_number', 'item_id']
).reset_index(drop=True)

# Numerical features
numeric_features = [
        'quantity_ordered',
        'year',
        'month',
        'quarter',
        'previous_paid_price',
        'previous_3_order_avg',
        'previous_3_order_median',
        'historical_item_avg',
        'historical_item_median',
        'historical_item_count',
        'historical_vendor_item_avg',
        'historical_vendor_item_median',
        'historical_vendor_item_count',
        'has_previous_vendor_item_history'
]

# Categorical features
categorical_features = [
    'item_id',
    'vendor_id'
]

# All features
FEATURES = numeric_features + categorical_features

# Convert numerical features
model_data[numeric_features] = (
    model_data[numeric_features]
    .apply(pd.to_numeric, errors='coerce')
)

# Convert target
model_data[TARGET] = pd.to_numeric(
    model_data[TARGET],
    errors='coerce'
)

# Remove rows without target
model_data = model_data.dropna(
    subset=[TARGET]
).copy()

print("Modelling records:", len(model_data))
print("Unique items:", model_data['item_id'].nunique())
print("Unique vendors:", model_data['vendor_id'].nunique())

display(model_data[FEATURES + [TARGET]].head())

Modelling records: 7905
Unique items: 129
Unique vendors: 12


,quantity_ordered,year,month,quarter,previous_paid_price,previous_3_order_avg,previous_3_order_median,historical_item_avg,historical_item_median,historical_item_count,historical_vendor_item_avg,historical_vendor_item_median,historical_vendor_item_count,has_previous_vendor_item_history,item_id,vendor_id,unit_price_paid
0,6,2024,1,1,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0,ITM-0434,VEND-10,1.39
1,6,2024,1,1,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0,ITM-0228,VEND-09,4.21
2,10,2024,1,1,NaN,NaN,NaN,1.39,1.39,1.00,NaN,NaN,0.00,0,ITM-0434,VEND-09,1.62
3,5,2024,1,1,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0,ITM-0929,VEND-09,1.18
4,6,2024,1,1,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0,ITM-0201,VEND-08,4.09


In [16]:
X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET].copy()

X_test = company_a_test[FEATURES].copy()
y_test = company_a_test[TARGET].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (6323, 16)
X_test: (416, 16)
y_train: (6323,)
y_test: (416,)


In [17]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [18]:
ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=10.0))
])

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    ))
])

In [19]:
models = {
    'Ridge': ridge_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

print("All models trained successfully.")

Training Ridge...


Training Random Forest...

Training XGBoost...


All models trained successfully.


In [20]:
baseline_predictions = (
    X_test['historical_vendor_item_avg'].copy()
)

# Fallback 1: Historical item average
baseline_predictions = baseline_predictions.fillna(
    X_test['historical_item_avg']
)

# Fallback 2: Training median
training_median = y_train.median()

baseline_predictions = baseline_predictions.fillna(
    training_median
)

#Predictions for all the models
predictions = {
    'Baseline': baseline_predictions,
    'Ridge': ridge_model.predict(X_test),
    'Random Forest': rf_model.predict(X_test),
    'XGBoost': xgb_model.predict(X_test)
}

In [21]:
def evaluate_model(y_actual, y_pred):

    y_actual = np.asarray(y_actual)
    y_pred = np.asarray(y_pred)

    absolute_errors = np.abs(y_actual - y_pred)

    return {
        'MAE': mean_absolute_error(y_actual, y_pred),

        'RMSE': np.sqrt(
            mean_squared_error(y_actual, y_pred)
        ),

        'MAPE (%)': np.mean(
            np.abs((y_actual - y_pred) / y_actual)
        ) * 100,

        'R2': r2_score(y_actual, y_pred),

        'Median AE': median_absolute_error(
            y_actual, y_pred
        ),

        'WAPE (%)': (
            absolute_errors.sum() /
            np.abs(y_actual).sum()
        ) * 100,

        'Bias': np.mean(y_pred - y_actual)
    }

In [22]:
results = {}

for name, prediction in predictions.items():

    results[name] = evaluate_model(
        y_test,
        prediction
    )

evaluation_results = pd.DataFrame(results).T

display(
    evaluation_results.round(4)
)

,MAE,RMSE,MAPE (%),R2,Median AE,WAPE (%),Bias
Baseline,0.28,1.73,3.54,0.98,0.07,4.11,-0.06
Ridge,0.52,2.13,14.03,0.97,0.27,7.57,0.19
Random Forest,0.43,2.15,4.82,0.96,0.09,6.29,0.15
XGBoost,0.47,2.22,7.85,0.96,0.13,6.79,0.14


In [23]:
# ============================================
# TOP 5 PREDICTION ERRORS: ALL FOUR MODELS
# ============================================

test_aligned = company_a_test.loc[X_test.index].copy()

top_errors = []

for model_name, prediction in predictions.items():

    diagnostics = test_aligned[
        [
            'customer_id',
            'item_id',
            'vendor_id',
            'po_number',
            'order_date',
            'quantity_ordered',
            'unit_price_paid'
        ]
    ].copy()

    diagnostics['prediction'] = np.asarray(prediction)

    diagnostics['absolute_error'] = (
        diagnostics['unit_price_paid'] -
        diagnostics['prediction']
    ).abs()

    diagnostics['Model'] = model_name

    # Select five largest errors
    model_top5 = diagnostics.nlargest(
        5, 'absolute_error'
    )

    top_errors.append(model_top5)


# Combine results
top_errors_all = pd.concat(
    top_errors,
    ignore_index=True
)

# Display separately by model
for model_name in predictions.keys():

    print(f"\n{'='*60}")
    print(f"TOP 5 ERRORS - {model_name}")
    print(f"{'='*60}")

    display(
        top_errors_all[
            top_errors_all['Model'] == model_name
        ][
            [
                'po_number',
                'item_id',
                'vendor_id',
                'order_date',
                'quantity_ordered',
                'unit_price_paid',
                'prediction',
                'absolute_error'
            ]
        ].sort_values(
            'absolute_error',
            ascending=False
        ).round(3)
    )


TOP 5 ERRORS - Baseline


,po_number,item_id,vendor_id,order_date,quantity_ordered,unit_price_paid,prediction,absolute_error
0,PO-101587,ITM-0902,VEND-08,2026-07-01,2,47.00,13.13,33.87
1,PO-102314,ITM-0902,VEND-11,2026-08-20,2,4.26,8.20,3.94
2,PO-102133,ITM-0250,VEND-09,2026-08-01,10,65.69,61.95,3.74
3,PO-101521,ITM-0715,VEND-07,2026-03-25,3,74.93,72.48,2.45
4,PO-100348,ITM-0250,VEND-11,2026-07-12,4,56.02,58.40,2.38



TOP 5 ERRORS - Ridge


,po_number,item_id,vendor_id,order_date,quantity_ordered,unit_price_paid,prediction,absolute_error
5,PO-101587,ITM-0902,VEND-08,2026-07-01,2,47.00,6.55,40.45
6,PO-100348,ITM-0250,VEND-11,2026-07-12,4,56.02,61.59,5.57
7,PO-102314,ITM-0902,VEND-11,2026-08-20,2,4.26,8.82,4.56
8,PO-100073,ITM-0902,VEND-09,2026-03-19,6,4.21,8.52,4.31
9,PO-101435,ITM-0902,VEND-09,2026-05-05,6,4.43,8.69,4.26



TOP 5 ERRORS - Random Forest


,po_number,item_id,vendor_id,order_date,quantity_ordered,unit_price_paid,prediction,absolute_error
10,PO-101587,ITM-0902,VEND-08,2026-07-01,2,47.00,6.02,40.98
11,PO-100348,ITM-0250,VEND-11,2026-07-12,4,56.02,61.54,5.52
12,PO-101448,ITM-0250,VEND-07,2026-06-10,10,60.18,64.97,4.79
13,PO-102301,ITM-0737,VEND-07,2026-07-14,1,72.18,76.45,4.27
14,PO-101521,ITM-0715,VEND-07,2026-03-25,3,74.93,71.16,3.77



TOP 5 ERRORS - XGBoost


,po_number,item_id,vendor_id,order_date,quantity_ordered,unit_price_paid,prediction,absolute_error
15,PO-101587,ITM-0902,VEND-08,2026-07-01,2,47.00,4.44,42.56
16,PO-102314,ITM-0902,VEND-11,2026-08-20,2,4.26,8.84,4.58
17,PO-100348,ITM-0250,VEND-11,2026-07-12,4,56.02,60.26,4.24
18,PO-101521,ITM-0715,VEND-07,2026-03-25,3,74.93,70.96,3.97
19,PO-101448,ITM-0250,VEND-07,2026-06-10,10,60.18,63.82,3.64


In [24]:
# Identify the highest-error item and compare its historical vendor pricing
item_diagnostics = test_data.loc[X_test.index, ["item_id", TARGET]].copy()
item_diagnostics["Random Forest_error"] = np.abs(item_diagnostics[TARGET].to_numpy() - np.asarray(predictions["Random Forest"]))
problem_item = item_diagnostics.groupby("item_id")["Random Forest_error"].mean().idxmax()
print("Highest-error item:", problem_item)

item_vendor_analysis = (
    model_data[model_data["item_id"] == problem_item]
    .groupby("vendor_id")
    .agg(Transaction_Count=(TARGET, "count"), Average_Price=(TARGET, "mean"), Minimum_Price=(TARGET, "min"), Maximum_Price=(TARGET, "max"), Median_Price=(TARGET, "median"))
    .reset_index()
)

display(item_vendor_analysis.round(2))

Highest-error item: ITM-0902


,vendor_id,Transaction_Count,Average_Price,Minimum_Price,Maximum_Price,Median_Price
0,VEND-08,30,13.99,4.02,48.50,4.53
1,VEND-09,38,6.52,4.04,47.80,4.44
2,VEND-10,32,11.65,3.99,48.00,4.36
3,VEND-11,21,7.83,3.81,43.80,4.26


In [25]:
# Display all historical transactions for the highest-error item
top_item_history = (
    model_data[model_data["item_id"] == problem_item]
    .sort_values(["order_date", "po_number"])
    .copy()
)
print("Item being investigated:", problem_item)
print("Total transactions:", len(top_item_history))
display(top_item_history[["customer_id", "order_date", "po_number", "vendor_id", "quantity_ordered", TARGET]])

Item being investigated: ITM-0902
Total transactions: 121


,customer_id,order_date,po_number,vendor_id,quantity_ordered,unit_price_paid
88,CUST-B,2024-01-10,PO-101966,VEND-10,1,4.31
181,CUST-B,2024-01-22,PO-100816,VEND-11,4,3.96
213,CUST-F,2024-01-25,PO-102147,VEND-08,1,4.32
269,CUST-B,2024-02-04,PO-100263,VEND-10,1,4.29
290,CUST-B,2024-02-08,PO-100517,VEND-09,1,4.13
...,...,...,...,...,...,...
7516,CUST-B,2026-07-18,PO-102608,VEND-09,1,4.90
7615,CUST-B,2026-07-29,PO-102124,VEND-10,12,4.91
7744,CUST-B,2026-08-13,PO-101977,VEND-08,2,4.87
7809,CUST-A,2026-08-20,PO-102314,VEND-11,2,4.26


In [26]:
# Analyse customer-vendor pricing for the highest-error item
customer_vendor_analysis = (
    top_item_history
    .groupby(["item_id", "customer_id", "vendor_id"])
    .agg(Transactions=(TARGET, "count"), 
         Average_Price=(TARGET, "mean"), 
         Median_Price=(TARGET, "median"), 
         Min_Price=(TARGET, "min"), 
         Max_Price=(TARGET, "max"))
    .reset_index()
)
display(customer_vendor_analysis.sort_values(["item_id", "customer_id", "Average_Price"]).round(2))

,item_id,customer_id,vendor_id,Transactions,Average_Price,Median_Price,Min_Price,Max_Price
3,ITM-0902,CUST-A,VEND-11,4,4.01,3.99,3.81,4.26
1,ITM-0902,CUST-A,VEND-09,6,4.22,4.20,4.04,4.43
0,ITM-0902,CUST-A,VEND-08,5,12.74,4.20,4.02,47.00
2,ITM-0902,CUST-A,VEND-10,7,13.96,4.26,3.99,39.00
5,ITM-0902,CUST-B,VEND-09,13,4.43,4.41,4.13,4.90
7,ITM-0902,CUST-B,VEND-11,8,9.20,4.30,3.96,43.80
6,ITM-0902,CUST-B,VEND-10,13,10.77,4.43,4.20,46.40
4,ITM-0902,CUST-B,VEND-08,15,12.90,4.51,4.21,48.50
10,ITM-0902,CUST-C,VEND-10,3,4.31,4.26,4.25,4.43
9,ITM-0902,CUST-C,VEND-09,3,16.43,4.60,4.49,40.20
